<img src="../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../images/Konstanz_Logo.svg" width="200" /> <img src="../images/KIT_Logo.png" width="200" />

# Introduction to DataFrames and Series

**pandas** is the standard Python library for working with **tabular data** — data
that comes in rows and columns, like a spreadsheet or a database table. It reads
and writes large files in many formats, and gives you the tools to select, filter
and transform what is inside them.

Everything in pandas is built out of just two objects:

- a **Series** — a single column of values, one-dimensional
- a **DataFrame** — several Series side by side, two-dimensional

This notebook introduces both. We start by building them by hand from Python lists
and dictionaries, so that it is completely clear what they are made of, and then
switch to a real file of New York taxi trips to see how the same ideas work at a
useful size.

---

## Contents

1. [The Series](#series)
2. [The DataFrame](#dataframe)
3. [Reading a DataFrame from a file](#reading)
4. [Working with columns](#columns)
5. [Working with rows](#rows)
6. [Filtering with conditions](#filtering)
7. [Applying your own function](#apply)

<a id="series"></a>
## 1. The Series

A Series is one-dimensional — a single column of values. Like a NumPy array it has
a **numerical index**: 0, 1, 2 and so on.

In [1]:
import pandas as pd

data = [83000000, 67000000, 17000000]

series = pd.Series(data=data)
series

0    83000000
1    67000000
2    17000000
dtype: int64

### 1.1 Labels

On top of that numbering, a Series can carry a **label** for each value, and the
label may be any hashable type — a string, a date, whatever fits the data.

That is the whole difference to a plain array, and it is what makes pandas
convenient: you can ask for the entry called `'Germany'` instead of remembering
that Germany happens to sit in position 0.

> On what "hashable" means, see the
> [Python glossary](https://docs.python.org/3/glossary.html#term-hashable).

In [2]:
index = ['Germany', 'France', 'Netherlands']

series = pd.Series(data=data, index=index)
series

Germany        83000000
France         67000000
Netherlands    17000000
dtype: int64

In [3]:
print("by position, series.iloc[0]    :", series.iloc[0])
print("by label,    series['Germany'] :", series['Germany'])

by position, series.iloc[0]    : 83000000
by label,    series['Germany'] : 83000000


Both give the same value. `.iloc` always means *"by position"*, square brackets
with a label mean *"by name"* — a distinction that comes back throughout pandas.

### 1.2 Creating a Series from a dictionary

A Python dictionary is already a set of key–value pairs, which is exactly the shape
of a labelled Series. So pandas converts one directly: keys become labels, values
become data.

In [4]:
age = {'Pia': 20, 'Felix': 26}

pd.Series(age)

Pia      20
Felix    26
dtype: int64

### 1.3 Calculating with Series

When two Series are combined, pandas matches them up **by label**, not by position.
You never have to sort anything or check that both sides are in the same order.

In [8]:
year_1990 = {'Germany': 70000000, 'France': 50000000, 'Netherlands': 12000000}
year_2021 = {'Germany': 83000000, 'France': 67000000, 'Netherlands': 17000000,
             'Greece': 13000000}

inhabitants_1990 = pd.Series(year_1990)
inhabitants_2021 = pd.Series(year_2021)

inhabitants_1990

Germany        70000000
France         50000000
Netherlands    12000000
dtype: int64

That raises a question: what about a label that exists on only one side? `'Greece'`
appears in 2021 but not in 1990.

In [9]:
inhabitants_1990 / inhabitants_2021

France         0.746269
Germany        0.843373
Greece              NaN
Netherlands    0.705882
dtype: float64

The result for Greece is `NaN` — *not a number*, pandas' marker for a missing
value. `div()` lets you say what should be used in place of the missing entry.

In [10]:
inhabitants_1990.div(inhabitants_2021, fill_value=13000000)

France         0.746269
Germany        0.843373
Greece         1.000000
Netherlands    0.705882
dtype: float64

<a id="dataframe"></a>
## 2. The DataFrame

A DataFrame is two-dimensional: a **group of Series that share the same index**.
Each Series becomes one column, and each column has a name you can select it by.

The illustration below shows two Series being assembled into one DataFrame. Both
describe the same three countries, so they share an index, and the result is a
single table with one row per country:

<p style="text-align: center;"> Pandas-Series 1: </p>

| Index       | Inhabitants 2021 |
|-------------|------------------|
| Germany     | 83000000         |
| France      | 67000000         |
| Netherlands | 17000000         |

<p style="text-align: center;"> Pandas-Series 2: </p>

| Index       | Inhabitants 1990 |
|-------------|------------------|
| Germany     | 70000000         |
| France      | 50000000         |
| Netherlands | 12000000         |

<p style="text-align: center;"> Pandas-DataFrame assembled from the two Pandas-Series:</p>

| Index       | Inhabitants 1990 | Inhabitants 2021 |
|-------------|------------------|------------------|
| Germany     | 70000000         | 83000000         |
| France      | 50000000         | 67000000         |
| Netherlands | 12000000         | 17000000         |

### 2.1 Building a DataFrame by hand

We start from a plain NumPy matrix of random numbers — four rows, three columns,
and no meaning attached to any of it.

In [11]:
import numpy as np

np.random.seed(42)                          # fixed seed, so the numbers stay the same
data = np.random.randint(0, 101, (4, 3))    # 4x3 matrix, random integers from 0 to 100
data

array([[51, 92, 14],
       [71, 60, 20],
       [82, 86, 74],
       [74, 87, 99]])

On its own that matrix says nothing: no column names, no row names. Handing it to
`pd.DataFrame` together with an `index` and `columns` turns it into a table you can
read.

In [12]:
index = ['Berlin', 'BW', 'Bayern', 'Hessen']
columns = ['Jan', 'Feb', 'Mar']

df = pd.DataFrame(data=data, index=index, columns=columns)
df

,Jan,Feb,Mar
Berlin,51,92,14
BW,71,60,20
Bayern,82,86,74
Hessen,74,87,99


In [13]:
df.info()      # dtypes, memory use, and how many values are non-empty

<class 'pandas.DataFrame'>
Index: 4 entries, Berlin to Hessen
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Jan     4 non-null      int64
 1   Feb     4 non-null      int64
 2   Mar     4 non-null      int64
dtypes: int64(3)
memory usage: 148.0 bytes


<a id="reading"></a>
## 3. Reading a DataFrame from a file

Building tables by hand is fine for three countries. In practice the data comes from
a file, and pandas reads most formats you are likely to meet — CSV, Excel, JSON,
SQL, Parquet.

Here we use a **Parquet** file of New York City taxi trips: one row per trip, with
pickup time, distance, fare, tip and so on.

*Data source: [https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)*

In [14]:
df = pd.read_parquet('../files/green_tripdata_2023-01.parquet', engine='pyarrow')
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00


The next few commands are the ones worth running on **any** table you open for the
first time — how big is it, what is in it, do the numbers look plausible.

In [15]:
print("Rows:   ", df.shape[0])
print("Columns:", df.shape[1])
print()
print(list(df.columns))

Rows:    68211
Columns: 20

['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge']


In [16]:
df.index       # the row labels -- here just 0, 1, 2, ...

RangeIndex(start=0, stop=68211, step=1)

In [17]:
df.describe().transpose().round(2)      # count, mean, min, max and quartiles

,count,mean,min,25%,50%,75%,max,std
VendorID,68211.0,1.863028,1.0,2.0,2.0,2.0,2.0,0.34382
lpep_pickup_datetime,68211,2023-01-16 20:10:55.679523,2009-01-01 20:21:27,2023-01-09 11:59:47.500000,2023-01-17 08:40:42,2023-01-24 15:52:30,2023-02-01 03:10:05,NaN
lpep_dropoff_datetime,68211,2023-01-16 20:29:01.515767,2009-01-02 11:07:31,2023-01-09 12:16:37.500000,2023-01-17 08:56:38,2023-01-24 16:06:56,2023-02-01 17:27:05,NaN
RatecodeID,63887.0,1.11716,1.0,1.0,1.0,1.0,99.0,1.372913
PULocationID,68211.0,98.549735,1.0,74.0,75.0,129.0,265.0,61.244314
DOLocationID,68211.0,138.429901,1.0,74.0,138.0,219.0,265.0,76.761311
passenger_count,63887.0,1.31587,0.0,1.0,1.0,1.0,9.0,0.979054
trip_distance,68211.0,8.114852,0.0,1.11,1.85,3.21,120098.84,585.105955
fare_amount,68211.0,16.603545,-70.0,9.3,13.5,19.8,490.0,13.470121
extra,68211.0,0.825431,-2.5,0.0,0.0,1.0,12.5,1.269904


<a id="columns"></a>
## 4. Working with columns

Selecting a single column gives back a **Series** — which is the point of section 1:
a DataFrame really is a collection of Series, and pulling one out returns the
original object.

In [18]:
df['tip_amount']

0        4.03
1        2.64
2        1.94
3        1.70
4        0.00
         ... 
68206    0.00
68207    0.00
68208    3.51
68209    3.20
68210    2.00
Name: tip_amount, Length: 68211, dtype: float64

In [19]:
type(df['tip_amount'])

pandas.Series

Ask for *several* columns and you get a DataFrame again, because more than one
Series side by side is exactly what a DataFrame is.

Note the double brackets: the outer ones select from `df`, the inner ones are the
Python list of names.

In [20]:
cols = ['tip_amount', 'total_amount']

df[cols]

,tip_amount,total_amount
0,4.03,24.18
1,2.64,15.84
2,1.94,11.64
3,1.70,10.20
4,0.00,8.00
...,...,...
68206,0.00,16.70
68207,0.00,5.41
68208,3.51,21.04
68209,3.20,19.18


In [21]:
type(df[cols])

pandas.DataFrame

Columns can be calculated with directly. The operation applies to every row at
once — there is no loop to write.

In [22]:
100 * df['tip_amount'] / df['total_amount']      # tip as a percentage of the fare

0        16.666667
1        16.666667
2        16.666667
3        16.666667
4         0.000000
           ...    
68206     0.000000
68207     0.000000
68208    16.682510
68209    16.684046
68210     6.802721
Length: 68211, dtype: float64

Assigning such a result to a new name **adds a column**.

In [23]:
df['tip_percentage'] = 100 * df['tip_amount'] / df['total_amount']
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,tip_percentage
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,...,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75,16.666667
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,...,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00,16.666667
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,...,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00,16.666667
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,...,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00,16.666667
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,...,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00,0.000000


Removing one is less obvious, and worth stepping through slowly. `drop()` does
**not** change the DataFrame — it returns a modified copy and leaves the original
untouched.

In [24]:
df.drop('tip_percentage', axis=1).head()      # a copy without the column

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00


In [25]:
df.head()                                    # ... but df itself still has it

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,tip_percentage
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,...,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75,16.666667
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,...,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00,16.666667
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,...,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00,16.666667
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,...,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00,16.666667
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,...,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00,0.000000


Only by assigning that copy back to `df` does the column really disappear. Most
pandas methods behave this way.

In [26]:
df = df.drop('tip_percentage', axis=1)
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00


`shape` reports the size of the table as `(rows, columns)`.

In [27]:
print("shape:  ", df.shape)
print("rows:   ", df.shape[0])
print("columns:", df.shape[1])

shape:   (68211, 20)
rows:    68211
columns: 20


<a id="rows"></a>
## 5. Working with rows

So far the index has been the automatic numbering. Any column can take over that
role instead, which is useful when the rows have a natural name — here, the moment
the trip started.

In [28]:
df.index

RangeIndex(start=0, stop=68211, step=1)

In [29]:
df = df.set_index("lpep_pickup_datetime")
df.head()

,VendorID,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
lpep_pickup_datetime,,,,,,,,,,,,,,,,,,,
2023-01-01 00:26:10,2,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
2023-01-01 00:51:03,2,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2023-01-01 00:35:12,2,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
2023-01-01 00:13:14,1,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
2023-01-01 00:33:04,1,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00


Careful: `lpep_pickup_datetime` is now the index and no longer a column.
`reset_index()` undoes it — again as a copy, so `df` itself is unchanged.

In [30]:
df.reset_index().head()

,lpep_pickup_datetime,VendorID,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2023-01-01 00:26:10,2,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
1,2023-01-01 00:51:03,2,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2,2023-01-01 00:35:12,2,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
3,2023-01-01 00:13:14,1,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
4,2023-01-01 00:33:04,1,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00


With rows there are two ways to select, mirroring the two we saw for a Series:
`iloc` works by **position**, `loc` works by **label**.

In [31]:
df.iloc[0:6]                # by position: the first six rows

,VendorID,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
lpep_pickup_datetime,,,,,,,,,,,,,,,,,,,
2023-01-01 00:26:10,2,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
2023-01-01 00:51:03,2,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,1.0,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00
2023-01-01 00:35:12,2,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00
2023-01-01 00:13:14,1,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,0.5,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00
2023-01-01 00:33:04,1,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,0.5,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00
2023-01-01 00:53:31,2,2023-01-01 01:11:04,N,1.0,41,262,1.0,2.78,17.7,1.0,0.5,0.00,0.0,None,1.0,22.95,2.0,1.0,2.75


In [32]:
df.loc[[pd.to_datetime('2023-01-01 00:26:10'),
        pd.to_datetime('2023-01-01 00:35:12')]]     # by label: exactly these two

,VendorID,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
lpep_pickup_datetime,,,,,,,,,,,,,,,,,,,
2023-01-01 00:26:10,2,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
2023-01-01 00:35:12,2,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,1.0,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00


`drop` removes rows as well as columns — `axis` decides which. `axis=0` means
rows, `axis=1` means columns.

In [33]:
print("before:", df.shape[0], "rows")
print("after: ", df.drop(pd.to_datetime('2023-01-01 00:26:10'), axis=0).shape[0], "rows")

before: 68211 rows
after:  68210 rows


A single row pulled out of a DataFrame is, once again, a **Series** — this time
with the column names as its labels.

In [34]:
row = df.iloc[0]
row

VendorID                                   2
lpep_dropoff_datetime    2023-01-01 00:37:11
store_and_fwd_flag                         N
RatecodeID                               1.0
PULocationID                             166
DOLocationID                             143
passenger_count                          1.0
trip_distance                           2.58
fare_amount                             14.9
extra                                    1.0
mta_tax                                  0.5
tip_amount                              4.03
tolls_amount                             0.0
ehail_fee                               None
improvement_surcharge                    1.0
total_amount                           24.18
payment_type                             1.0
trip_type                                1.0
congestion_surcharge                    2.75
Name: 2023-01-01 00:26:10, dtype: object

To attach it to the end of the table, `pd.concat` glues DataFrames together. The
row has to be turned back into a one-row DataFrame first, which is what
`.to_frame().T` does.

In [35]:
df = pd.concat([df, row.to_frame().T])
df.tail(3)

,VendorID,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
2023-01-31 23:01:00,2,2023-01-31 23:19:00,NaN,NaN,225,189,NaN,3.03,14.98,0.0,0.0,3.2,0.0,None,1.0,19.18,NaN,NaN,NaN
2023-01-31 23:51:00,2,2023-02-01 00:07:00,NaN,NaN,256,140,NaN,5.82,23.65,0.0,0.0,2.0,0.0,None,1.0,29.4,NaN,NaN,NaN
2023-01-01 00:26:10,2,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,1.0,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75


<a id="filtering"></a>
## 6. Filtering with conditions

A note on vocabulary first: every **column** is called a *feature* of the data, and
every **row** is called an *instance*.

We read the file again here, because the index was replaced in the previous section
and we want the plain numbering back.

In [36]:
df = pd.read_parquet('../files/green_tripdata_2023-01.parquet', engine='pyarrow')
df.index

RangeIndex(start=0, stop=68211, step=1)

Comparing a column against a value does not give you rows — it gives a column of
`True` and `False`, one entry per row. This is called a **boolean mask**.

In [37]:
df['tip_amount'] > 3            # where is the tip larger than 3?

0         True
1        False
2        False
3        False
4        False
         ...  
68206    False
68207    False
68208     True
68209     True
68210    False
Name: tip_amount, Length: 68211, dtype: bool

Putting that mask back into the square brackets keeps only the rows where it says
`True`.

In [38]:
large_tip = df['tip_amount'] > 3

df[large_tip]

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.90,1.00,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75
6,1,2023-01-01 00:09:14,2023-01-01 00:26:39,N,1.0,181,45,2.0,3.80,19.10,3.75,1.5,4.85,0.0,None,1.0,29.20,1.0,1.0,2.75
11,2,2023-01-01 00:08:43,2023-01-01 00:17:08,N,1.0,75,140,1.0,1.99,11.40,1.00,0.5,3.33,0.0,None,1.0,19.98,1.0,1.0,2.75
13,2,2023-01-01 00:18:35,2023-01-01 00:30:09,N,1.0,66,255,1.0,3.23,17.70,1.00,0.5,4.04,0.0,None,1.0,24.24,1.0,1.0,0.00
17,2,2023-01-01 00:35:11,2023-01-01 01:17:35,N,1.0,97,68,1.0,8.28,44.30,1.00,0.5,5.00,0.0,None,1.0,54.55,1.0,1.0,2.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68201,2,2023-01-31 20:57:00,2023-01-31 21:17:00,NaN,NaN,166,68,NaN,6.92,27.74,0.00,0.0,3.15,0.0,None,1.0,34.64,NaN,NaN,NaN
68203,2,2023-01-31 20:33:00,2023-01-31 20:52:00,NaN,NaN,166,141,NaN,3.31,15.65,0.00,0.0,3.88,0.0,None,1.0,23.28,NaN,NaN,NaN
68204,2,2023-01-31 21:53:00,2023-01-31 22:05:00,NaN,NaN,42,236,NaN,2.26,14.62,0.00,0.0,3.31,0.0,None,1.0,21.68,NaN,NaN,NaN
68208,2,2023-01-31 23:46:00,2023-02-01 00:02:00,NaN,NaN,66,37,NaN,3.44,16.53,0.00,0.0,3.51,0.0,None,1.0,21.04,NaN,NaN,NaN


Masks can be combined: `&` means *and*, `|` means *or*. Each condition needs its
own brackets, because `&` binds more tightly than `>` in Python.

In [39]:
df[(df['tip_amount'] > 3) & (df['total_amount'] > 40)]

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
17,2,2023-01-01 00:35:11,2023-01-01 01:17:35,N,1.0,97,68,1.0,8.28,44.30,1.0,0.5,5.00,0.00,None,1.0,54.55,1.0,1.0,2.75
31,2,2023-01-01 00:46:48,2023-01-01 01:08:17,N,4.0,95,265,1.0,10.44,47.10,1.0,0.5,9.92,0.00,None,1.0,59.52,1.0,1.0,0.00
32,2,2023-01-01 00:40:58,2023-01-01 01:04:32,N,5.0,66,164,4.0,6.78,60.00,0.0,0.0,12.61,0.00,None,0.3,75.66,1.0,2.0,2.75
49,2,2023-01-01 00:23:04,2023-01-01 23:18:32,N,1.0,255,48,1.0,8.55,47.80,1.0,0.5,5.30,0.00,None,1.0,58.35,1.0,1.0,2.75
75,2,2023-01-01 01:23:35,2023-01-01 01:53:43,N,1.0,255,249,1.0,4.90,29.60,1.0,0.5,8.71,0.00,None,1.0,43.56,1.0,1.0,2.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68180,2,2023-01-31 17:11:00,2023-01-31 17:48:00,NaN,NaN,244,265,NaN,19.08,80.82,0.0,0.0,16.96,3.00,None,1.0,101.78,NaN,NaN,NaN
68182,2,2023-01-31 17:34:00,2023-01-31 18:32:00,NaN,NaN,166,132,NaN,18.62,77.00,0.0,0.0,16.91,6.55,None,1.0,101.46,NaN,NaN,NaN
68185,2,2023-01-31 17:22:00,2023-01-31 17:48:00,NaN,NaN,193,234,NaN,4.78,25.26,0.0,0.0,7.11,6.55,None,1.0,42.67,NaN,NaN,NaN
68195,2,2023-01-31 19:52:00,2023-01-31 20:24:00,NaN,NaN,40,238,NaN,9.61,34.28,0.0,0.0,6.85,0.00,None,1.0,44.88,NaN,NaN,NaN


`isin` tests against a whole list of values at once — much shorter than writing
`(col == 2) | (col == 5)`.

In [40]:
wanted = [2, 5]

df['passenger_count'].isin(wanted).iloc[95:105]     # the mask for rows 95 to 104

95     False
96     False
97     False
98      True
99     False
100    False
101     True
102     True
103     True
104     True
Name: passenger_count, dtype: bool

In [41]:
print("passenger_count in row 95:", df.iloc[95]['passenger_count'])  # -> not 2 or 5

passenger_count in row 95: 1.0


In [42]:
df.info()      # a reminder of what the columns hold

<class 'pandas.DataFrame'>
RangeIndex: 68211 entries, 0 to 68210
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               68211 non-null  int64         
 1   lpep_pickup_datetime   68211 non-null  datetime64[us]
 2   lpep_dropoff_datetime  68211 non-null  datetime64[us]
 3   store_and_fwd_flag     63887 non-null  str           
 4   RatecodeID             63887 non-null  float64       
 5   PULocationID           68211 non-null  int64         
 6   DOLocationID           68211 non-null  int64         
 7   passenger_count        63887 non-null  float64       
 8   trip_distance          68211 non-null  float64       
 9   fare_amount            68211 non-null  float64       
 10  extra                  68211 non-null  float64       
 11  mta_tax                68211 non-null  float64       
 12  tip_amount             68211 non-null  float64       
 13  tolls_amount

<a id="apply"></a>
## 7. Applying your own function

When the transformation you need is not built into pandas, write an ordinary Python
function and hand it to `apply`. It is called once per value in the column.

The example takes the last two digits of a number.

In [43]:
def last_two(number):
    return int(str(number)[-2:])


print(last_two(1234567))       # check it on a single value first

67


In [44]:
df['PULocationID'].apply(last_two)      # ... then on the whole column

0        66
1        24
2        23
3        41
4        41
         ..
68206    49
68207    10
68208    66
68209    25
68210    56
Name: PULocationID, Length: 68211, dtype: int64

The result is a Series like any other, so it can be stored as a new column.

In [45]:
df['PULocationID_last_two'] = df['PULocationID'].apply(last_two)
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,PULocationID_last_two
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.0,166,143,1.0,2.58,14.9,...,0.5,4.03,0.0,None,1.0,24.18,1.0,1.0,2.75,66
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.0,24,43,1.0,1.81,10.7,...,0.5,2.64,0.0,None,1.0,15.84,1.0,1.0,0.00,24
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.0,223,179,1.0,0.00,7.2,...,0.5,1.94,0.0,None,1.0,11.64,1.0,1.0,0.00,23
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.0,41,238,1.0,1.30,6.5,...,1.5,1.70,0.0,None,1.0,10.20,1.0,1.0,0.00,41
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.0,41,74,1.0,1.10,6.0,...,1.5,0.00,0.0,None,1.0,8.00,1.0,1.0,0.00,41


And columns come with the usual summary methods built in.

In [46]:
print(f"mean total amount: {df['total_amount'].mean():.2f}")

mean total amount: 21.79
